---
## Bagian 10 — Terapkan pada data proyek tim

**Ini bagian yang dinilai.** Hasilnya menjadi bagian pertama dari **M2 — Data Quality Report**
yang dikumpulkan pada pertemuan 5.

Kerjakan bersama tim pada dataset proyek Anda sendiri.

In [6]:
# 10.1 — muat dataset proyek tim
# df_tim = pd.read_csv('data/raw/....csv')
# df_tim.info()
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

FILE = 'SuperMarket Analysis.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df_tim = pd.read_csv(FILE)
df_tim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   object 
 1   Branch                   1000 non-null   object 
 2   City                     1000 non-null   object 
 3   Customer type            1000 non-null   object 
 4   Gender                   1000 non-null   object 
 5   Product line             1000 non-null   object 
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Sales                    1000 non-null   float64
 10  Date                     1000 non-null   object 
 11  Time                     1000 non-null   object 
 12  Payment                  1000 non-null   object 
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  

In [ ]:
# 10.2 — jalankan kesembilan pemeriksaan pada data tim
# Salin pola Bagian 2 sampai 8. Untuk setiap pemeriksaan, catat temuannya.
# 1. Potret awal: shape, info, describe

# 2. Nilai kosong sejati
# 3. Placeholder teks dan angka
# 4. Pola kekosongan: MCAR / MAR / MNAR untuk setiap kolom yang banyak kosong
# 5. Duplikat persis dan duplikat menurut kunci
# 6. Konsistensi kolom teks
# 7. Tipe data yang salah
# 8. Aturan rentang
# 9. Aturan hubungan antarkolom


In [4]:
print("Jumlah baris dan kolom:", df_tim.shape)
print("\nTipe data setiap kolom:")
print(df_tim.dtypes)

print("\nStatistik deskriptif:")
display(df_tim.describe(include='all').T)

Jumlah baris dan kolom: (1000, 17)

Tipe data setiap kolom:
Invoice ID                  object
Branch                      object
City                        object
Customer type               object
Gender                      object
Product line                object
Unit price                 float64
Quantity                     int64
Tax 5%                     float64
Sales                      float64
Date                        object
Time                        object
Payment                     object
cogs                       float64
gross margin percentage    float64
gross income               float64
Rating                     float64
dtype: object

Statistik deskriptif:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Invoice ID,1000,1000,849-09-3807,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Branch,1000,3,Alex,340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,1000,3,Yangon,340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer type,1000,2,Member,565,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,1000,2,Female,571,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product line,1000,6,Fashion accessories,178,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unit price,1000.0,NaN,NaN,NaN,55.67213,26.494628,10.08,32.875,55.23,77.935,99.96
Quantity,1000.0,NaN,NaN,NaN,5.51,2.923431,1.0,3.0,5.0,8.0,10.0
Tax 5%,1000.0,NaN,NaN,NaN,15.379369,11.708825,0.5085,5.924875,12.088,22.44525,49.65
Sales,1000.0,NaN,NaN,NaN,322.966749,245.885335,10.6785,124.422375,253.848,471.35025,1042.65


**PEMERIKSAAN 1: Potret awal: shape, info, describe**

Temuan: Dataset memiliki 1.000 baris dan 17 kolom. Statistik deskriptif digunakan untuk melihat distribusi data, nilai minimum dan maksimum, serta karakteristik kolom numerik dan kategorikal.

In [7]:
missing = pd.DataFrame({
    'Jumlah Kosong': df_tim.isna().sum(),
    'Persentase Kosong (%)':
        (df_tim.isna().sum() / len(df_tim) * 100).round(2)
})

display(missing)

print("Total nilai kosong:", df_tim.isna().sum().sum())

,Jumlah Kosong,Persentase Kosong (%)
Invoice ID,0,0.0
Branch,0,0.0
City,0,0.0
Customer type,0,0.0
Gender,0,0.0
Product line,0,0.0
Unit price,0,0.0
Quantity,0,0.0
Tax 5%,0,0.0
Sales,0,0.0


Total nilai kosong: 0


**PEMERIKSAAN 2: Nilai kosong sejati**

Temuan: Tidak ditemukan nilai kosong sejati atau missing value pada seluruh kolom dataset. Dengan demikian, tidak ada kolom yang memiliki persentase nilai kosong di atas 5%.

In [8]:
placeholders = [
    '', 'na', 'n/a', 'null', 'none', 'unknown', '-'
]

for col in df_tim.columns:
    data = df_tim[col].astype(str).str.strip()

    jumlah_placeholder = data.str.lower().isin(
        placeholders
    ).sum()

    if jumlah_placeholder > 0:
        print(
            f"{col}: {jumlah_placeholder} placeholder"
        )

    if df_tim[col].dtype == 'object':
        jumlah_spasi = (
            df_tim[col].astype(str) !=
            df_tim[col].astype(str).str.strip()
        ).sum()

        if jumlah_spasi > 0:
            print(
                f"{col}: {jumlah_spasi} nilai memiliki "
                "spasi di awal atau akhir"
            )

**PEMERIKSAAN 3: Placeholder teks dan angka**

Temuan: Tidak ditemukan placeholder umum seperti NA, N/A, null, none, unknown, atau -. Pemeriksaan juga tidak menemukan spasi di awal maupun akhir nilai teks. Dengan catatan, pemeriksaan ini mendeteksi placeholder yang telah ditentukan, bukan semua kemungkinan nilai tidak valid.

**PEMERIKSAAN 4: Pola kekosongan: MCAR / MAR / MNAR untuk setiap kolom yang banyak kosong**

Temuan: Tidak terdapat nilai kosong pada dataset, sehingga tidak ada kolom dengan persentase missing value di atas 5%. Oleh karena itu, klasifikasi MCAR, MAR, dan MNAR tidak diperlukan untuk dataset ini.

In [9]:
# Duplikat persis
duplikat_persis = df_tim.duplicated().sum()

print("Jumlah duplikat persis:", duplikat_persis)

# Duplikat berdasarkan Invoice ID
if 'Invoice ID' in df_tim.columns:
    duplikat_invoice = df_tim['Invoice ID'].duplicated().sum()

    print("Jumlah Invoice ID duplikat:", duplikat_invoice)

    print(
        "Jumlah Invoice ID unik:",
        df_tim['Invoice ID'].nunique()
    )

Jumlah duplikat persis: 0
Jumlah Invoice ID duplikat: 0
Jumlah Invoice ID unik: 1000


**PEMERIKSAAN 5: Duplikat persis dan duplikat menurut kunci**

Temuan: Hasil pemeriksaan menunjukkan tidak terdapat baris duplikat persis dan tidak terdapat Invoice ID yang berulang. Seluruh 1.000 transaksi memiliki identitas faktur yang unik. Berdasarkan pemeriksaan tersebut, tidak ditemukan indikasi duplikasi pada data transaksi.

In [10]:
# Memeriksa nilai unik setiap kolom teks
kolom_teks = df_tim.select_dtypes(
    include='object'
).columns

for col in kolom_teks:
    print(f"\nKolom: {col}")
    print("Jumlah nilai unik:", df_tim[col].nunique())
    print(sorted(df_tim[col].dropna().unique()))

# Memeriksa hubungan Branch dan City
if {'Branch', 'City'}.issubset(df_tim.columns):
    print("\nHubungan Branch dan City:")
    display(pd.crosstab(df_tim['Branch'], df_tim['City']))


Kolom: Invoice ID
Jumlah nilai unik: 1000
['101-17-6199', '101-81-4070', '102-06-2002', '102-77-2261', '105-10-6182', '105-31-1824', '106-35-6779', '109-28-2512', '109-86-4363', '110-05-6330', '110-48-7033', '114-35-5271', '115-38-7388', '115-99-4379', '118-62-1812', '120-06-4233', '120-54-2248', '122-61-9553', '123-19-1176', '123-35-4896', '124-31-1458', '125-45-2293', '126-54-1082', '127-47-6963', '129-29-8530', '130-67-4723', '130-98-8941', '131-15-8856', '131-70-8179', '132-23-6451', '132-32-9879', '133-14-7229', '133-77-3154', '134-54-4720', '134-75-2619', '135-13-8269', '135-84-8019', '136-08-6195', '137-63-5492', '137-74-8729', '138-17-5109', '139-20-0155', '139-32-4183', '139-52-2867', '142-63-6033', '142-72-4741', '144-51-6085', '145-94-9061', '146-09-5432', '148-41-7930', '148-82-2527', '149-14-0304', '149-15-7606', '149-61-1929', '149-71-6266', '150-89-8043', '151-16-1484', '151-27-8496', '151-33-7434', '152-03-4217', '152-08-9985', '152-68-2907', '153-58-4872', '154-74-717

City,Mandalay,Naypyitaw,Yangon
Branch,,,
Alex,0,0,340
Cairo,332,0,0
Giza,0,328,0


**PEMERIKSAAN 6: Konsistensi kolom teks**

Temuan: Nilai kategorikal pada kolom Branch, City, Customer type, Gender, Product line, dan Payment konsisten dalam ejaan dan kapitalisasi. Pemeriksaan hubungan cabang dan kota menunjukkan pasangan Alex–Yangon, Cairo–Mandalay, dan Giza–Naypyitaw.
Tidak ditemukan indikasi inkonsistensi kategori teks berdasarkan pemeriksaan nilai unik dan hubungan antarkolom tersebut.

In [11]:
print("Tipe data awal:")
print(df_tim.dtypes)

# Memeriksa format tanggal
tanggal = pd.to_datetime(
    df_tim['Date'],
    errors='coerce'
)

print(
    "\nJumlah tanggal gagal diparse:",
    tanggal.isna().sum()
)

# Memeriksa format waktu
waktu = pd.to_datetime(
    df_tim['Time'],
    format='%I:%M:%S %p',
    errors='coerce'
)

print(
    "Jumlah waktu gagal diparse:",
    waktu.isna().sum()
)

Tipe data awal:
Invoice ID                  object
Branch                      object
City                        object
Customer type               object
Gender                      object
Product line                object
Unit price                 float64
Quantity                     int64
Tax 5%                     float64
Sales                      float64
Date                        object
Time                        object
Payment                     object
cogs                       float64
gross margin percentage    float64
gross income               float64
Rating                     float64
dtype: object

Jumlah tanggal gagal diparse: 0
Jumlah waktu gagal diparse: 0


**PEMERIKSAAN 7: Tipe data yang salah**

Temuan: Kolom Date dan Time masih memiliki tipe data object, padahal keduanya secara semantik merepresentasikan tanggal dan waktu. Pemeriksaan format dilakukan untuk mengetahui apakah nilainya dapat dibaca sebagai tanggal dan waktu yang valid.
Kedua kolom tersebut merupakan kandidat perubahan tipe data pada tahap perbaikan berikutnya. Pada tahap ini, data tidak diubah.

In [12]:
aturan_rentang = {
    'Unit price > 0':
        df_tim['Unit price'] > 0,

    'Quantity bilangan bulat dan >= 1':
        (df_tim['Quantity'] >= 1) &
        (df_tim['Quantity'] % 1 == 0),

    'Rating berada pada skala 0-10':
        df_tim['Rating'].between(0, 10),

    'Tax 5% tidak negatif':
        df_tim['Tax 5%'] >= 0,

    'Sales tidak negatif':
        df_tim['Sales'] >= 0,

    'cogs tidak negatif':
        df_tim['cogs'] >= 0,

    'gross income tidak negatif':
        df_tim['gross income'] >= 0
}

for nama, kondisi in aturan_rentang.items():
    jumlah_pelanggaran = (~kondisi).sum()

    print(
        f"{nama}: {jumlah_pelanggaran} pelanggaran"
    )

Unit price > 0: 0 pelanggaran
Quantity bilangan bulat dan >= 1: 0 pelanggaran
Rating berada pada skala 0-10: 0 pelanggaran
Tax 5% tidak negatif: 0 pelanggaran
Sales tidak negatif: 0 pelanggaran
cogs tidak negatif: 0 pelanggaran
gross income tidak negatif: 0 pelanggaran


**PEMERIKSAAN 8: Aturan rentang**

Temuan: Pemeriksaan dilakukan menggunakan tujuh aturan validitas yang disesuaikan dengan karakteristik transaksi supermarket. Harga satuan harus lebih besar dari nol, kuantitas harus berupa bilangan bulat minimal satu, rating harus berada pada rentang 0–10, dan nilai pajak, biaya barang, penjualan, serta pendapatan kotor tidak boleh negatif.
Hasil pemeriksaan menunjukkan tidak ditemukan pelanggaran pada aturan rentang yang digunakan. Batas rating 0–10 merupakan asumsi validitas yang perlu disesuaikan apabila dokumentasi dataset menetapkan skala berbeda.

In [13]:
aturan_hubungan = {
    'cogs = Unit price * Quantity':
        np.isclose(
            df_tim['cogs'],
            df_tim['Unit price'] * df_tim['Quantity']
        ),

    'Tax 5% = 5% * cogs':
        np.isclose(
            df_tim['Tax 5%'],
            df_tim['cogs'] * 0.05
        ),

    'Sales = cogs + Tax 5%':
        np.isclose(
            df_tim['Sales'],
            df_tim['cogs'] + df_tim['Tax 5%']
        ),

    'gross income = Tax 5%':
        np.isclose(
            df_tim['gross income'],
            df_tim['Tax 5%']
        )
}

for nama, kondisi in aturan_hubungan.items():
    jumlah_pelanggaran = (~kondisi).sum()

    print(
        f"{nama}: {jumlah_pelanggaran} pelanggaran"
    )

cogs = Unit price * Quantity: 0 pelanggaran
Tax 5% = 5% * cogs: 0 pelanggaran
Sales = cogs + Tax 5%: 0 pelanggaran
gross income = Tax 5%: 0 pelanggaran


**PEMERIKSAAN 9: Aturan hubungan antarkolom**

Temuan: Pemeriksaan hubungan antarkolom dilakukan untuk memastikan konsistensi perhitungan transaksi. Biaya barang atau cogs seharusnya sama dengan harga satuan dikalikan kuantitas. Pajak sebesar 5% dihitung berdasarkan biaya barang, sedangkan total penjualan merupakan penjumlahan biaya barang dan pajak.
Selain itu, kolom gross income diperiksa terhadap nilai pajak. Hasil pemeriksaan menunjukkan seluruh hubungan tersebut sesuai pada 1.000 baris data, tanpa pelanggaran berdasarkan aturan yang digunakan.

In [14]:
# 10.3 — susun profil kualitas data tim
# Pakai fungsi catat() yang sama, lalu simpan.
# temuan_tim = []
# ...
# profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False)
# Menyimpan ringkasan sembilan pemeriksaan

temuan_tim = [
    {
        'Pemeriksaan': '1. Potret awal',
        'Temuan': (
            'Dataset memiliki 1000 baris dan 17 kolom. '
            'Kolom Date dan Time bertipe object.'
        )
    },
    {
        'Pemeriksaan': '2. Nilai kosong sejati',
        'Temuan': (
            'Tidak ditemukan missing value pada seluruh kolom.'
        )
    },
    {
        'Pemeriksaan': '3. Placeholder teks dan angka',
        'Temuan': (
            'Tidak ditemukan placeholder umum atau spasi '
            'di awal maupun akhir nilai teks.'
        )
    },
    {
        'Pemeriksaan': '4. MCAR, MAR, dan MNAR',
        'Temuan': (
            'Tidak ada kolom dengan missing value di atas 5%, '
            'sehingga klasifikasi tidak diperlukan.'
        )
    },
    {
        'Pemeriksaan': '5. Duplikat',
        'Temuan': (
            'Tidak ditemukan duplikat persis maupun '
            'Invoice ID duplikat.'
        )
    },
    {
        'Pemeriksaan': '6. Konsistensi teks',
        'Temuan': (
            'Kategori teks konsisten. Pasangan Branch dan '
            'City juga konsisten.'
        )
    },
    {
        'Pemeriksaan': '7. Tipe data',
        'Temuan': (
            'Kolom Date dan Time bertipe object dan menjadi '
            'kandidat konversi tipe pada tahap perbaikan.'
        )
    },
    {
        'Pemeriksaan': '8. Aturan rentang',
        'Temuan': (
            'Tidak ditemukan pelanggaran berdasarkan tujuh '
            'aturan validitas yang diterapkan.'
        )
    },
    {
        'Pemeriksaan': '9. Hubungan antarkolom',
        'Temuan': (
            'Seluruh hubungan perhitungan yang diperiksa '
            'sesuai tanpa pelanggaran.'
        )
    }
]

profil_tim = pd.DataFrame(temuan_tim)

# Membuat folder reports jika belum tersedia
os.makedirs('reports', exist_ok=True)

# Menyimpan profil kualitas data
profil_tim.to_csv(
    'reports/profil_kualitas_data.csv',
    index=False
)

display(profil_tim)

print(
    "\nProfil kualitas data berhasil disimpan di "
    "reports/profil_kualitas_data.csv"
)


,Pemeriksaan,Temuan
0,1. Potret awal,Dataset memiliki 1000 baris dan 17 kolom. Kolo...
1,2. Nilai kosong sejati,Tidak ditemukan missing value pada seluruh kolom.
2,3. Placeholder teks dan angka,Tidak ditemukan placeholder umum atau spasi di...
3,"4. MCAR, MAR, dan MNAR",Tidak ada kolom dengan missing value di atas 5...
4,5. Duplikat,Tidak ditemukan duplikat persis maupun Invoice...
5,6. Konsistensi teks,Kategori teks konsisten. Pasangan Branch dan C...
6,7. Tipe data,Kolom Date dan Time bertipe object dan menjadi...
7,8. Aturan rentang,Tidak ditemukan pelanggaran berdasarkan tujuh ...
8,9. Hubungan antarkolom,Seluruh hubungan perhitungan yang diperiksa se...



Profil kualitas data berhasil disimpan di reports/profil_kualitas_data.csv


### Daftar periksa Bagian 10

- [ ] Seluruh sembilan pemeriksaan dijalankan pada data tim
- [ ] Setiap kolom yang punya nilai kosong di atas 5% sudah dinilai MCAR, MAR, atau MNAR, **beserta alasannya**
- [ ] Minimal lima aturan validitas ditulis sendiri berdasarkan pengetahuan tentang data tim
- [ ] Minimal satu aturan hubungan antarkolom diperiksa
- [ ] Profil kualitas data tersimpan di `reports/`
- [ ] Notebook di-*commit* ke repositori tim dengan pesan yang bermakna

> **Ingat:** hari ini **tidak ada** yang diperbaiki. Perbaikan beserta justifikasinya adalah pekerjaan
> pertemuan 5. Yang diserahkan hari ini adalah diagnosis.